# [Final project](https://github.com/hanggrian/IIT-CS587/blob/assets/assignments/proj_1.pdf): Experiments

## [Instructions](https://github.com/hanggrian/IIT-CS587/blob/assets/assignments/proj_2.pdf)

> Build a Virtual Team with AI Agents to Develop Software Project Plan

```mermaid
flowchart LR
    CP[Customer Proxy] <--> PM[Project Manager]
    PM --> CE[Requirements Engineer]
    PM --> SE[System Engineer]
    PM --> SD[Software Developer]
    PM --> TE[Test Engineer]
    PM --> DE[Documentation Engineer]
```

> For your final project, you will create a project plan using GenAI technologies, libraries, and frameworks as discussed in the class. You will be creating a project plan for a selected software application from any of the following domains:
>
> - Retail
> - Financial
> - Healthcare
> - Manufacturing
> - Energy
> - Education
> - Real Estate
> - Food Services
> - Transportation
> - Insurance
> - etc.

![Strickland Propane](https://github.com/hanggrian/IIT-CS587/raw/assets/proj/image.jpg)

The selected software application is an Enterprise Resource Planning system for Strickland Propane, a fictional company that sells propane and propane accessories. They operate in the retail and energy sectors.

## Setup

OpenAI models are chosen for this experiment. Compared to Llama and DeepSeek, OpenAI models provide concise answers and are more well-documented.

In [ ]:
from os import getenv
from dotenv import load_dotenv, find_dotenv

def get_openai_api_key():
    _ = load_dotenv(find_dotenv())
    openai_api_key = getenv('OPENAI_API_KEY')
    return openai_api_key

llm_config = {
    'model': 'gpt-4o-mini',
    'api_key': get_openai_api_key(),
}

Describe the project plan and its required features. Then, provide supporting documents that can be fed to a Microsoft Project file.

In [2]:
task = \
    'You are a software project team tasked to create an Enterprise Resource Planning project for Strickland Propane business with features such as Customer Relationship Management, Inventory Management, Ordering and Billing. ' + \
    'Given extensive features, the software project is considered complex, but should not take longer than several months to complete. ' + \
    'Devise a Work Breakdown Structure in Microsoft Project format using the effort estimation by agent engineers. '

## Phase 1: Waterfall

Insert the software methodology plan in tasks.

In [3]:
task += 'Assume waterfall software development methodology.'

Define seven agents with different roles. Ask engineers how long should it take for them to complete three sections:

- The main task assigned to the role.
- Review the work.
- Rework when necessary.

In [4]:
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager

# customer and manager
cp = \
    UserProxyAgent(
        name='CP',
        description='Customer Proxy',
        human_input_mode='NEVER',
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        code_execution_config={
            'last_n_messages': 1,
            'work_dir': 'tasks',
            'use_docker': False,
        },
    )
pm = \
    AssistantAgent(
        name='PM',
        description='Project Manager',
        llm_config=llm_config,
        system_message= \
            'You are a project manager, a natural leader who excels in team organization. ' + \
            'You monitor project progress, set deadlines and manage the budget. ' + \
            'You take immediate action to overcome obstacles using your creative problem-solving skills.',
    )

# engineers
re = \
    AssistantAgent(
        name='RE',
        description='Requirements Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a requirements engineer who analyzes software requirements. ' + \
            'You are responsible for creating a report according to the requirements. ' + \
            'Most importantly, you are to attend meetings with clients and project managers to collect these requirements.',
    )
se = \
    AssistantAgent(
        name='SE',
        description='Systems Engineer',
        llm_config=llm_config,
        system_message= \
            "You are a system engineer capable of architecting complex software systems that meet client's requirements. " + \
            'When the system is running, you are required to troubleshoot errors and patch security threats.',
    )
sd = \
    AssistantAgent(
        name='SD',
        description='Software Developer',
        llm_config=llm_config,
        system_message= \
            'You are a software developer tasked with implementing and maintaining software programs. ' + \
            'This activity involves producing a new source code and integrating existing software components. ' + \
            'However, the bulk of the testing is performed by a test engineer.',
    )
te = \
    AssistantAgent(
        name='TE',
        description='Test Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a test engineer, working closely with other engineers to identify software defects. ' + \
            'You are an expert in testing frameworks for executing test plans to ensure the software meets quality standards.',
    )
de = \
    AssistantAgent(
        name='DE',
        description='Documentation Engineer',
        llm_config=llm_config,
        system_message= \
            'You are a documentation engineer who creates supporting documents to help others keep track of their work. ' + \
            'These documents often include visual graphics and technical manuals.',
    )

# chats
chat_manager = \
    GroupChatManager(
        groupchat= \
            GroupChat(
            agents=[cp, pm, re, se, sd, te, de],
            messages=[],
            speaker_selection_method='round_robin',
            allow_repeat_speaker=False,
            max_round=1,
        ),
        is_termination_msg=lambda x: x.get('content', '').find('TERMINATE') >= 0,
        llm_config=llm_config,
        code_execution_config={
            'work_dir': 'coding',
            'use_docker': False,
        },
    )

Start the chat!

In [ ]:
def any_message(section_title, sections):
    message = f"Append a new section '{section_title}' with sub-sections:\n\n"
    for _, section in enumerate(sections):
        message += f'- {section}\n'
    message += '\nPredict the amount of effort required for each section.'
    return message

def pm_message(recipient, messages, sender, config):
    return any_message(
        'Project plan',
        ['Write initial plan', 'Review initial plan', 'Rework'],
    )
def re_message(recipient, messages, sender, config):
    return any_message(
        'Requirements',
        ['Write requirements', 'Review requirements', 'Rework'],
    )
def se_message(recipient, messages, sender, config):
    return any_message(
        'Design document and data model',
        ['Write design documents', 'Write data models'],
    )
def sd_message(recipient, messages, sender, config):
    return any_message(
        'Coding and unit test',
        ['Write code', 'Unit testing', 'Code inspection'],
    )
def te_message(recipient, messages, sender, config):
    return any_message(
        'Testing',
        ['Write test plans', 'Execute test plans', 'Fix found defects'],
    )
def de_message(recipient, messages, sender, config):
    return any_message(
        'Documentation',
        ['Write user documentation', 'Review user documentation'],
    )

pm.register_nested_chats(
    [
        {
            'recipient': chat_manager,
            'summary_method': 'reflection_with_llm',
            'clear_history': False,
        },
        {
            'recipient': pm,
            'message': pm_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': re,
            'message': re_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': se,
            'message': se_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': sd,
            'message': sd_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': te,
            'message': te_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': de,
            'message': de_message,
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
        {
            'recipient': chat_manager,
            'message': 'Devise a Work Breakdown Structure using the estimation by engineers.',
            'summary_method': 'last_msg',
            'max_turns': 1,
        },
    ],
    trigger=cp,
)
cp.initiate_chats([
    {
        'recipient': pm,
        'message': task,
        'max_turns': 1,
        'summary_method': 'last_msg',
    }
])


********************************************************************************
Starting a new chat....

********************************************************************************
CP (to PM):

You are a software project team tasked to create an Enterprise Resource Planning project for Strickland Propane business with features such as Customer Relationship Management, Inventory Management, Ordering and Billing. Given extensive features, the software project is considered complex, but should not take longer than several months to complete. Devise a Work Breakdown Structure in Microsoft Project format using the effort estimation by agent engineers. Assume waterfall software development methodology.

--------------------------------------------------------------------------------

********************************************************************************
Starting a new chat....

********************************************************************************
PM (to chat_manag

/home/hanggrian/GitHub/IIT-CS587/proj/.venv/lib/python3.10/site-packages/autogen/agentchat/chat.py:47: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(



********************************************************************************
Starting a new chat....

********************************************************************************
PM (to PM):

Append a new section 'Project plan' with sub-sections:

- Write initial plan
- Review initial plan
- Rework

Predict the amount of effort required for each section.
Context: 
The task is to create a Work Breakdown Structure (WBS) for an Enterprise Resource Planning project for Strickland Propane, incorporating features like Customer Relationship Management, Inventory Management, and Ordering and Billing. The project follows a waterfall methodology and should be completed in several months, with effort estimations by engineers taken into account.

--------------------------------------------------------------------------------
PM (to PM):

### Project Plan

#### Write Initial Plan
**Effort Required:** 3 weeks (120 hours)

- **Activities:**
  - Define project scope and objectives
  - Gather

[ChatResult(chat_id=None, chat_history=[{'content': 'You are a software project team tasked to create an Enterprise Resource Planning project for Strickland Propane business with features such as Customer Relationship Management, Inventory Management, Ordering and Billing. Given extensive features, the software project is considered complex, but should not take longer than several months to complete. Devise a Work Breakdown Structure in Microsoft Project format using the effort estimation by agent engineers. Assume waterfall software development methodology.', 'role': 'assistant', 'name': 'CP'}, {'content': 'Devise a Work Breakdown Structure using the estimation by engineers.\nContext: \nThe task is to create a Work Breakdown Structure (WBS) for an Enterprise Resource Planning project for Strickland Propane, incorporating features like Customer Relationship Management, Inventory Management, and Ordering and Billing. The project follows a waterfall methodology and should be completed in

## Phase 1: Scrum

Insert the software methodology plan in tasks.

In [ ]:
task += 'Assume scrum software development methodology.'